<a href="https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why
I chose Random Forest because my task is to identify content that may need attention. It can use multiple features together and may find patterns that the simple Week-4 rule misses.

I will use F1 score to compare the model because the main class I want to identify is content with a downward trend.

In [12]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)

## 2. Split design
The Week-4 baseline was a fixed rule and did not use a train/test split. For this model, I use an 80/20 split grouped by client.

This helps keep content from the same client from appearing in both training and test data. I will use the same test data to compare the model with the Week-4 baseline.

In [13]:
DATA_PATH = Path("/content/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Clients:", df["client_id"].nunique())

df["target_down"] = (
    df["trend_direction"] == "down"
).astype(int)

print("\nTarget distribution:")
print(df["target_down"].value_counts())

Shape: (30000, 44)
Clients: 32

Target distribution:
target_down
1    16262
0    13738
Name: count, dtype: int64


In [14]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

feature_columns = numeric_features + categorical_features

X = df[feature_columns]
y = df["target_down"]
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())

print(
    "Client overlap:",
    len(
        set(df.iloc[train_idx]["client_id"])
        &
        set(df.iloc[test_idx]["client_id"])
    )
)

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

I use the Week-4 baseline as a fixed comparison rule. A baseline score of 3 or higher means the content is marked for attention, while a score below 3 means monitor.

I train the Random Forest only on the training data. Both the baseline and the model are evaluated on the same test data using F1, precision, recall, and balanced accuracy.

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_features
        )
    ]
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)

model_pred = pipeline.predict(X_test)

print("F1:", round(f1_score(y_test, model_pred), 4))
print("Precision:", round(precision_score(y_test, model_pred), 4))
print("Recall:", round(recall_score(y_test, model_pred), 4))
print(
    "Balanced accuracy:",
    round(
        balanced_accuracy_score(y_test, model_pred),
        4
    )
)

F1: 0.5913
Precision: 0.5927
Recall: 0.59
Balanced accuracy: 0.5832


In [16]:
test = df.iloc[test_idx].copy()

staleness_score_map = {
    "0-30": 0,
    "31-90": 1,
    "91-180": 2,
    "181+": 3
}

volume_score_map = {
    "low": 0,
    "moderate": 1,
    "good": 2,
    "excellent": 3
}

test["staleness_score"] = test["freshness_tier"].map(
    staleness_score_map
)

test["volume_score"] = test["impression_tier"].map(
    volume_score_map
)

test["baseline_score"] = (
    test["staleness_score"]
    + test["volume_score"]
)

baseline_pred = (
    test["baseline_score"] >= 3
).astype(int)

print("Baseline F1:",
      round(
          f1_score(
              test["target_down"],
              baseline_pred
          ),
          4
      ))

print("Baseline Precision:",
      round(
          precision_score(
              test["target_down"],
              baseline_pred
          ),
          4
      ))

print("Baseline Recall:",
      round(
          recall_score(
              test["target_down"],
              baseline_pred
          ),
          4
      ))

print(
    "Baseline Balanced Accuracy:",
    round(
        balanced_accuracy_score(
            test["target_down"],
            baseline_pred
        ),
        4
    )
)

Baseline F1: 0.1881
Baseline Precision: 0.4565
Baseline Recall: 0.1185
Baseline Balanced Accuracy: 0.4856


In [17]:
comparison = pd.DataFrame([
    {
        "method": "Week-4 baseline",
        "f1": f1_score(
            test["target_down"],
            baseline_pred
        ),
        "precision": precision_score(
            test["target_down"],
            baseline_pred
        ),
        "recall": recall_score(
            test["target_down"],
            baseline_pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            test["target_down"],
            baseline_pred
        )
    },
    {
        "method": "Random Forest",
        "f1": f1_score(
            y_test,
            model_pred
        ),
        "precision": precision_score(
            y_test,
            model_pred
        ),
        "recall": recall_score(
            y_test,
            model_pred
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            model_pred
        )
    }
])

display(comparison.round(4))

,method,f1,precision,recall,balanced_accuracy
0,Week-4 baseline,0.1881,0.4565,0.1185,0.4856
1,Random Forest,0.5913,0.5927,0.5900,0.5832


## 4. Errors and interpretation

The Random Forest performed better than the Week-4 baseline on the held-out test data. Its F1 score was 0.5913 compared with 0.1881 for the baseline. I inspected the model errors to understand which cases it gets wrong.

The model made both false-positive and false-negative predictions, so it is not perfect. Some actual down-trending content was missed, including cases with large negative trend percentages.

Permutation importance showed that days_with_impressions was the strongest observed feature, followed by impressions_90d, avg_position, and position_tier. This suggests that the model relied mainly on visibility and search-position related signals.

Overall, the Random Forest is useful as a decision-support model, but the errors show that its predictions should still be reviewed before taking action.

In [18]:
cm = confusion_matrix(
    y_test,
    model_pred
)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        model_pred,
        target_names=["not_down", "down"]
    )
)

Confusion Matrix:
[[1737 1277]
 [1291 1858]]

Classification Report:
              precision    recall  f1-score   support

    not_down       0.57      0.58      0.57      3014
        down       0.59      0.59      0.59      3149

    accuracy                           0.58      6163
   macro avg       0.58      0.58      0.58      6163
weighted avg       0.58      0.58      0.58      6163



In [19]:
error_analysis = df.iloc[test_idx].copy()

error_analysis["prediction"] = model_pred

error_analysis["error_type"] = np.select(
    [
        (error_analysis["target_down"] == 0)
        & (error_analysis["prediction"] == 1),

        (error_analysis["target_down"] == 1)
        & (error_analysis["prediction"] == 0)
    ],
    [
        "false_positive",
        "false_negative"
    ],
    default="correct"
)

print("False positives:")
display(
    error_analysis[
        error_analysis["error_type"] == "false_positive"
    ].head(10)
)

print("\nFalse negatives:")
display(
    error_analysis[
        error_analysis["error_type"] == "false_negative"
    ].head(10)
)

False positives:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,target_down,prediction,error_type
13,content_a5a2fbc76336,client_8527a891e2,10.0,0.0,LOW,0.0,keyword article,informational,1342.0,8469.0,...,0.00,25.00,0.0,moderate,page_3_5,stable,10.4,0,1,false_positive
26,content_72c5c2d73e5a,client_4e07408562,0.0,0.0,LOW,0.0,keyword article,informational,2686.0,17181.0,...,0.00,11.11,0.0,moderate,page_3_5,stable,-9.2,0,1,false_positive
36,content_bce275871a25,client_f369cb89fc,0.0,0.0,LOW,0.0,keyword article,informational,2510.0,15518.0,...,0.00,0.00,0.0,moderate,page_1,stable,-18.9,0,1,false_positive
64,content_685de0e3b7cb,client_f369cb89fc,10.0,0.0,LOW,0.0,keyword article,informational,2808.0,19244.0,...,0.00,0.00,0.0,moderate,page_1,up,45.2,0,1,false_positive
78,content_dea0d86223f3,client_8527a891e2,0.0,0.0,LOW,0.0,keyword article,informational,1589.0,10358.0,...,0.00,0.00,0.0,low,page_1,up,1400.0,0,1,false_positive
82,content_ec6fce716c78,client_4e07408562,20.0,0.0,LOW,0.0,keyword article,commercial,2793.0,17593.0,...,12.50,8.33,0.0,moderate,page_1,up,41.7,0,1,false_positive
126,content_be5e23c0a35e,client_f369cb89fc,0.0,0.0,LOW,0.0,keyword article,informational,2492.0,20337.0,...,0.00,0.00,0.0,low,page_3_5,up,127.3,0,1,false_positive
135,content_670746e86425,client_4e07408562,20.0,0.0,LOW,0.0,keyword article,commercial,2844.0,17399.0,...,12.75,13.68,0.0,good,page_3_5,up,28.5,0,1,false_positive
179,content_552a9396d8dc,client_8527a891e2,20.0,0.0,LOW,0.0,keyword article,informational,3393.0,19822.0,...,0.00,0.00,0.0,moderate,striking,up,49.5,0,1,false_positive
181,content_722d8cd002d3,client_f369cb89fc,0.0,0.0,LOW,0.0,keyword article,informational,2602.0,17587.0,...,0.00,16.67,0.0,moderate,striking,up,46.8,0,1,false_positive



False negatives:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,target_down,prediction,error_type
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.0,10.0,0.0,good,page_3_5,down,-57.7,1,0,false_negative
23,content_2da6ae9d0882,client_e629fa6598,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,25.0,0.0,low,striking,down,-100.0,1,0,false_negative
25,content_033ae3e7aecf,client_f369cb89fc,70.0,0.81,HIGH,1.12,keyword article,commercial,2777.0,16215.0,...,0.0,0.0,50.0,low,page_1,down,-35.7,1,0,false_negative
39,content_4595e8704e07,client_8527a891e2,90.0,0.06,LOW,0.03,keyword article,informational,3666.0,21824.0,...,0.0,0.0,0.0,low,page_3_5,down,-100.0,1,0,false_negative
47,content_40cb4af260c0,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,3086.0,23347.0,...,0.0,50.0,0.0,low,page_3_5,down,-83.3,1,0,false_negative
51,content_d8a23b5e10c5,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,2756.0,23316.0,...,0.0,200.0,0.0,low,page_1,down,-100.0,1,0,false_negative
54,content_ff8ea1364b59,client_e629fa6598,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,100.0,0.0,low,striking,down,-88.0,1,0,false_negative
60,content_b9104a222d01,client_f369cb89fc,30.0,0.85,HIGH,0.29,keyword article,transactional,2492.0,14794.0,...,0.0,0.0,0.0,low,page_1,down,-33.3,1,0,false_negative
81,content_16788821b64a,client_e629fa6598,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,0.0,20.0,moderate,page_1,down,-20.8,1,0,false_negative
90,content_abfa53fe7911,client_e629fa6598,50.0,0.02,LOW,0.01,keyword article,informational,NaN,NaN,...,20.0,20.0,0.0,low,striking,down,-85.7,1,0,false_negative


In [20]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    pipeline,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
})

importance = importance.sort_values(
    "importance_mean",
    ascending=False
)

display(importance.head(15))

,feature,importance_mean,importance_std
13,days_with_impressions,0.029863,0.005361
5,impressions_90d,0.012920,0.002431
19,avg_position,0.008048,0.002186
33,position_tier,0.006288,0.001686
4,char_count,0.004463,0.001321
32,impression_tier,0.003236,0.001116
3,word_count,0.003223,0.001374
6,clicks_90d,0.003052,0.001179
29,freshness_tier,0.002618,0.001191
21,scroll_rate,0.001168,0.001093


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.